In [5]:
# === SETUP: load the provided data files (regenerate them if missing) ===
import os
import numpy as np
import pandas as pd

def build_datasets(csv_path='ecommerce_customers.csv',
                   xlsx_path='transactions.xlsx', seed=42, verbose=False):
    """Generate a realistic e-commerce customer + transactions dataset.

    Baked-in realism for EDA / feature engineering practice:
      - right-skewed monetary columns (total_spend) with a few 'whale' outliers
      - class imbalance in is_churned (~20-30%)
      - real signal: churn depends on recency, order count and support tickets
      - missing values in age / gender / city
      - a high-cardinality 'city' column (long tail of rare cities)
      - customers with zero orders (no last_order_date) -> dormant
    The Excel file is order-level and is CONSISTENT with the customer table
    (num_orders / total_spend / last_order_date are derived from it).
    """
    rng = np.random.default_rng(seed)
    N = 2500
    start = pd.Timestamp('2021-01-01')
    end = pd.Timestamp('2024-06-30')
    horizon = (end - start).days

    cust = np.array([f'CUST{i+1:05d}' for i in range(N)])
    signup_off = rng.integers(0, horizon - 60, N)
    signup = start + pd.to_timedelta(signup_off, unit='D')

    # order counts: overdispersed (gamma-poisson), some customers have zero
    lam = rng.gamma(2.0, 1.6, N)
    num_orders = rng.poisson(lam)

    # ---- order-level transactions (vectorised) ----
    counts = num_orders
    tot = int(counts.sum())
    cust_rep = np.repeat(cust, counts)
    signup_rep = np.repeat(signup_off, counts)
    span = np.maximum(horizon - signup_off, 1)
    span_rep = np.repeat(span, counts)
    off = (rng.random(tot) * span_rep).astype(int)
    tx_off = signup_rep + off
    tx_date = start + pd.to_timedelta(tx_off, unit='D')
    amount = rng.lognormal(3.2, 0.8, tot).round(2)        # right-skewed (~tens of currency)
    category = rng.choice(['Electronics', 'Fashion', 'Grocery', 'Home', 'Books'],
                          tot, p=[.20, .30, .25, .15, .10])
    tx = pd.DataFrame({'customer_id': cust_rep, 'order_date': tx_date,
                       'amount': amount, 'category': category}).sort_values(
        ['customer_id', 'order_date']).reset_index(drop=True)

    # ---- aggregate transactions -> customer level ----
    agg = tx.groupby('customer_id').agg(
        total_spend=('amount', 'sum'),
        first_order=('order_date', 'min'),
        last_order=('order_date', 'max'),
    ).reset_index()

    df = pd.DataFrame({'customer_id': cust, 'signup_date': signup,
                       'num_orders': num_orders})
    df = df.merge(agg, on='customer_id', how='left')
    df['total_spend'] = df['total_spend'].fillna(0).round(2)

    # ---- demographics & account attributes ----
    df['age'] = np.clip(rng.normal(38, 12, N), 18, 82).round().astype(int)
    df['gender'] = rng.choice(['M', 'F', 'Other'], N, p=[.48, .48, .04])

    majors = ['Mumbai', 'Delhi', 'Bengaluru', 'Hyderabad', 'Chennai', 'Pune', 'Kolkata']
    rare = ['Jaipur', 'Surat', 'Indore', 'Bhopal', 'Patna', 'Nagpur',
            'Kochi', 'Coimbatore', 'Visakhapatnam', 'Lucknow']
    pool = majors + rare
    w = np.array([.17, .15, .14, .12, .10, .08, .06] + [.013] * 10)
    w = w / w.sum()
    df['city'] = rng.choice(pool, N, p=w)

    df['plan'] = rng.choice(['Basic', 'Standard', 'Premium'], N, p=[.50, .35, .15])
    df['device'] = rng.choice(['Mobile', 'Desktop', 'Tablet'], N, p=[.60, .32, .08])
    df['payment_method'] = rng.choice(['Card', 'UPI', 'Wallet', 'NetBanking'],
                                      N, p=[.40, .35, .15, .10])
    df['support_tickets'] = rng.poisson(0.6, N)
    df['email_opt_in'] = rng.choice([0, 1], N, p=[.35, .65])

    # ---- churn target with real signal (recency / orders / tickets) ----
    last = pd.to_datetime(df['last_order'])
    days_since = (end - last).dt.days
    days_since_filled = days_since.fillna(horizon).to_numpy()
    z = (-2.75
         + 0.0019 * days_since_filled
         + 0.30 * df['support_tickets'].to_numpy()
         - 0.05 * df['num_orders'].to_numpy()
         + 0.70 * (df['num_orders'].to_numpy() == 0))
    p = 1 / (1 + np.exp(-z))
    df['is_churned'] = (rng.random(N) < p).astype(int)

    # ---- format dates as ISO strings (NaT -> ) ----
    df = df.rename(columns={'first_order': 'first_order_date',
                            'last_order': 'last_order_date'})
    for c in ['signup_date', 'first_order_date', 'last_order_date']:
        df[c] = pd.to_datetime(df[c]).dt.date.astype('string')

    df = df[['customer_id', 'signup_date', 'first_order_date', 'last_order_date',
             'age', 'gender', 'city', 'plan', 'device', 'payment_method',
             'num_orders', 'total_spend', 'support_tickets', 'email_opt_in',
             'is_churned']]

    # ---- inject missing values AFTER computing the target ----
    def punch(col, frac):
        idx = rng.choice(N, int(frac * N), replace=False)
        df.loc[idx, col] = np.nan
    punch('age', 0.07)
    punch('gender', 0.04)
    punch('city', 0.02)

    # ---- write files ----
    df.to_csv(csv_path, index=False)
    tx_out = tx.copy()
    tx_out['order_date'] = pd.to_datetime(tx_out['order_date']).dt.date.astype('string')
    tx_out.to_excel(xlsx_path, index=False)

    if verbose:
        print('customers:', df.shape, '| transactions:', tx_out.shape)
        print('churn rate:', round(df["is_churned"].mean(), 3))
        print('total_spend skew:', round(df["total_spend"].skew(), 2))
        print('missing age:', int(df["age"].isna().sum()),
              '| missing city:', int(df["city"].isna().sum()))
        print('zero-order customers:', int((df["num_orders"] == 0).sum()))
        print('distinct cities:', df["city"].nunique())
    return df, tx_out

if not (os.path.exists('ecommerce_customers.csv') and os.path.exists('transactions.xlsx')):
    build_datasets()
    print('Generated dataset files.')
else:
    print('Found the provided dataset files.')

Found the provided dataset files.


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

SNAPSHOT = pd.Timestamp('2024-06-30')   # the day the data was pulled

cust = pd.read_csv('ecommerce_customers.csv',
                   parse_dates=['signup_date', 'first_order_date', 'last_order_date'])
tx = pd.read_excel('transactions.xlsx', parse_dates=['order_date'])
print('customers:', cust.shape, '| transactions:', tx.shape)
tx.head()

In [ ]:
#1. Aggregate transactions → customer features (RFM)
# 🔹 1A. ROLL UP ORDER-LEVEL ROWS TO ONE ROW PER CUSTOMER
rfm = tx.groupby('customer_id').agg(
    frequency=('amount', 'size'),                 # F: how many orders
    monetary=('amount', 'sum'),                   # M: total spent
    avg_order_value=('amount', 'mean'),           # typical basket
    last_order=('order_date', 'max'),
    n_categories=('category', 'nunique'),         # breadth of interest
).reset_index()
# R: recency = days since last order, measured from the snapshot
rfm['recency_days'] = (SNAPSHOT - rfm['last_order']).dt.days
rfm[['customer_id', 'frequency', 'monetary', 'avg_order_value',
     'recency_days', 'n_categories']].head()

In [ ]:
# 🔹 1B. MERGE THE NEW FEATURES BACK ONTO THE CUSTOMER TABLE
df = cust.merge(rfm.drop(columns='last_order'), on='customer_id', how='left')
# customers with no transactions -> fill sensible defaults
df['frequency'] = df['frequency'].fillna(0)
df['monetary'] = df['monetary'].fillna(0)
df['n_categories'] = df['n_categories'].fillna(0)
# never ordered -> recency is 'infinitely' stale: cap at customer tenure
df['recency_days'] = df['recency_days'].fillna(9999)
print('Merged shape:', df.shape)
df[['customer_id', 'frequency', 'monetary', 'recency_days', 'is_churned']].head()

In [11]:

import pandas as pd
import numpy as np
# 1. Load the datasets
cust = pd.read_csv(
    'ecommerce_customers.csv',
    parse_dates=['signup_date', 'first_order_date', 'last_order_date']
)
tx = pd.read_excel(
    'transactions.xlsx',
    parse_dates=['order_date']
)

# 2. Create customer-level RFM features
rfm = tx.groupby('customer_id').agg(
    frequency=('amount', 'size'),
    monetary=('amount', 'sum'),
    avg_order_value=('amount', 'mean'),
    last_order=('order_date', 'max'),
    n_categories=('category', 'nunique')
).reset_index()

# 3. Merge RFM features into customer data
df = cust.merge(
    rfm.drop(columns='last_order'),
    on='customer_id',
    how='left'
)
df['frequency'] = df['frequency'].fillna(0)
df['monetary'] = df['monetary'].fillna(0)
df['n_categories'] = df['n_categories'].fillna(0)

# 4. LAB EXERCISE 1: Calculate first and last order dates
order_span = tx.groupby('customer_id')['order_date'].agg(
    ['min', 'max']
).reset_index()
# Calculate active span in days
order_span['active_span_days'] = (
    order_span['max'] - order_span['min']
).dt.days

# 5. Calculate average days between orders
order_span = order_span.merge(
    df[['customer_id', 'frequency']],
    on='customer_id',
    how='left'
)
order_span['avg_days_between_orders'] = np.where(
    order_span['frequency'] > 1,
    order_span['active_span_days'] /
    (order_span['frequency'] - 1),
    np.nan
)

# 6. Merge the new feature into df
df = df.merge(
    order_span[['customer_id', 'avg_days_between_orders']],
    on='customer_id',
    how='left',
    validate='one_to_one'
)

# 7. Display the result
print("Updated dataset shape:", df.shape)
df[['customer_id', 'frequency',
    'avg_days_between_orders']].head()


Updated dataset shape: (2500, 20)


,customer_id,frequency,avg_days_between_orders
0,CUST00001,0.0,NaN
1,CUST00002,0.0,NaN
2,CUST00003,1.0,NaN
3,CUST00004,1.0,NaN
4,CUST00005,4.0,212.0


In [ ]:
#2. Numeric transforms — fixing skew
#2A. LOG-TRANSFORM A RIGHT-SKEWED COLUMN
df['monetary_log'] = np.log1p(df['monetary'])   # log1p handles 0 safely
print('skew before:', round(df['monetary'].skew(), 2))
print('skew after :', round(df['monetary_log'].skew(), 2))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
ax[0].hist(df['monetary'], bins=40, color='#3AAFA9'); ax[0].set_title('monetary (raw)')
ax[1].hist(df['monetary_log'], bins=40, color='#2D6A9F'); ax[1].set_title('monetary (log1p)')
plt.tight_layout(); plt.show()

In [12]:
#LAB EXCERCISE-2
# 1. Fill missing average order values with 0
df['avg_order_value'] = df['avg_order_value'].fillna(0)
# Check skewness before transformation
skew_before = df['avg_order_value'].skew()

# 2. Apply log1p transformation
df['aov_log'] = np.log1p(df['avg_order_value'])

# 3. Check skewness after transformation
skew_after = df['aov_log'].skew()
print('Skew before:', round(skew_before, 2))
print('Skew after :', round(skew_after, 2))

Skew before: 2.72
Skew after : -1.42


In [ ]:
#3. Binning — continuous → groups
# 3A. BUCKET AGE INTO LIFE-STAGE GROUPS
df['age_group'] = pd.cut(df['age'], bins=[0, 25, 40, 60, 200],
                         labels=['<25', '25-39', '40-59', '60+'])
print(df['age_group'].value_counts(dropna=False))

In [24]:
#LAB EXCERCISE-3
import pandas as pd
import numpy as np
# 1. Load datasets
SNAPSHOT = pd.Timestamp('2024-06-30')

cust = pd.read_csv(
    'ecommerce_customers.csv',
    parse_dates=[
        'signup_date',
        'first_order_date',
        'last_order_date'
    ]
)
tx = pd.read_excel(
    'transactions.xlsx',
    parse_dates=['order_date']
)

# 2. Calculate RFM features from transactions
rfm = tx.groupby('customer_id').agg(
    frequency=('amount', 'size'),
    monetary=('amount', 'sum'),
    last_order=('order_date', 'max')
).reset_index()

# 3. Merge features into customer data
df = cust.merge(
    rfm,
    on='customer_id',
    how='left',
    validate='one_to_one'
)

# 4. Fill missing transaction values
df['frequency'] = df['frequency'].fillna(0)
df['monetary'] = df['monetary'].fillna(0)
# 5. Calculate recency in days
df['recency_days'] = (
    SNAPSHOT - df['last_order']
).dt.days
# For customers with no orders, use customer tenure
tenure = (SNAPSHOT - df['signup_date']).dt.days
df['recency_days'] = df['recency_days'].fillna(tenure)

# 6. LAB EXERCISE 3: Create recency buckets
df['recency_bucket'] = pd.cut(
    df['recency_days'],
    bins=[-1, 30, 90, float('inf')],
    labels=['active', 'lapsing', 'dormant'],
    include_lowest=True
)
# 7. Print customer counts
print("Customer counts per recency bucket:")
print(df['recency_bucket'].value_counts(sort=False, dropna=False))
# 8. Cross-tabulate recency bucket against churn
print("\nChurn distribution by recency bucket:")
churn_table = pd.crosstab(
    df['recency_bucket'],
    df['is_churned'],
    normalize='index'
).round(3)
print(churn_table)
# 9. Display churn rates as percentages
print("\nChurn rate (%) by recency bucket:")
print((churn_table[1] * 100).round(2))
# 10. Preview results
print("\nPreview:")
print(df[['customer_id', 'recency_days',
          'recency_bucket', 'is_churned']].head())


Customer counts per recency bucket:
recency_bucket
active      462
lapsing     543
dormant    1495
Name: count, dtype: int64

Churn distribution by recency bucket:
is_churned          0      1
recency_bucket              
active          0.933  0.067
lapsing         0.936  0.064
dormant         0.767  0.233

Churn rate (%) by recency bucket:
recency_bucket
active      6.7
lapsing     6.4
dormant    23.3
Name: 1, dtype: float64

Preview:
  customer_id  recency_days recency_bucket  is_churned
0   CUST00001        1168.0        dormant           0
1   CUST00002         335.0        dormant           1
2   CUST00003         405.0        dormant           0
3   CUST00004         729.0        dormant           0
4   CUST00005          15.0         active           0


In [32]:
#4. Ratios & interaction features
import pandas as pd
import numpy as np
# 1. Load datasets
cust = pd.read_csv('ecommerce_customers.csv')
tx = pd.read_excel('transactions.xlsx')
# 2. Calculate transaction features
rfm = tx.groupby('customer_id').agg(
    frequency=('amount', 'size'),
    monetary=('amount', 'sum'),
    n_categories=('category', 'nunique')
).reset_index()
# 3. Merge features with customer data
df = cust.merge(
    rfm,
    on='customer_id',
    how='left',
    validate='one_to_one'
)
# 4. Fill missing transaction values
for col in ['frequency', 'monetary', 'n_categories']:
    df[col] = df[col].fillna(0)
# 5. Calculate ratios — Exercise 4A
# Tickets per order
df['tickets_per_order'] = (
    df['support_tickets'] /
    df['frequency'].replace(0, np.nan)
).fillna(0)
# Spend per category
df['spend_per_category'] = (
    df['monetary'] /
    df['n_categories'].replace(0, np.nan)
).fillna(0)
# 6. Display results
print("Exercise 4A: Ratios & Interaction Features")
print(df[[
    'support_tickets',
    'frequency',
    'tickets_per_order',
    'monetary',
    'n_categories',
    'spend_per_category'
]].head())




Exercise 4A: Ratios & Interaction Features
   support_tickets  frequency  tickets_per_order  monetary  n_categories  \
0                2        0.0                0.0      0.00           0.0   
1                1        0.0                0.0      0.00           0.0   
2                1        1.0                1.0     39.17           1.0   
3                1        1.0                1.0      2.74           1.0   
4                0        4.0                0.0    133.80           3.0   

   spend_per_category  
0                0.00  
1                0.00  
2               39.17  
3                2.74  
4               44.60  


In [34]:

import pandas as pd
import numpy as np
# Define the snapshot date
SNAPSHOT = pd.Timestamp('2024-06-30')
# 1. Convert signup_date to datetime
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')
# Ensure frequency is numeric
df['frequency'] = pd.to_numeric(df['frequency'], errors='coerce').fillna(0)
# Calculate tenure in months
tenure_months = (
    (SNAPSHOT - df['signup_date']).dt.days / 30
)
# Calculate orders per month, guarding against zero tenure
df['orders_per_month'] = (
    df['frequency'] / tenure_months.replace(0, np.nan)
)

# 2. Replace infinite and missing values with 0
df['orders_per_month'] = (
    df['orders_per_month']
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# 3. Show the 5 highest-frequency customers
print(
    df.nlargest(5, 'frequency')[
        ['customer_id', 'signup_date', 'frequency', 'orders_per_month']
    ]
)


     customer_id signup_date  frequency  orders_per_month
237    CUST00238  2022-06-10       19.0          0.758988
812    CUST00813  2023-04-13       18.0          1.216216
562    CUST00563  2022-08-23       17.0          0.753323
1199   CUST01200  2023-01-25       16.0          0.919540
1918   CUST01919  2022-07-04       16.0          0.660248


In [35]:
#5.Datetime features
#  5A. UNLOCK THE DATE COLUMNS
df['tenure_days'] = (SNAPSHOT - df['signup_date']).dt.days   # how long a customer
df['signup_month'] = df['signup_date'].dt.month
df['signup_dow'] = df['signup_date'].dt.dayofweek            # 0=Mon
df['signup_is_weekend'] = (df['signup_dow'] >= 5).astype(int)
# cyclical encoding so Dec(12) and Jan(1) are 'close', not far apart
df['month_sin'] = np.sin(2 * np.pi * df['signup_month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['signup_month'] / 12)
df[['signup_date', 'tenure_days', 'signup_month',
    'signup_dow', 'signup_is_weekend', 'month_sin', 'month_cos']].head()

,signup_date,tenure_days,signup_month,signup_dow,signup_is_weekend,month_sin,month_cos
0,2021-04-19,1168,4,0,0,8.660254e-01,-5.000000e-01
1,2023-07-31,335,7,0,0,-5.000000e-01,-8.660254e-01
2,2023-03-07,481,3,1,0,1.000000e+00,6.123234e-17
3,2022-06-18,743,6,5,1,1.224647e-16,-1.000000e+00
4,2022-06-11,750,6,5,1,1.224647e-16,-1.000000e+00


In [36]:
# LAB EXERCISE 5 — DAYS SINCE FIRST ORDER
import pandas as pd
import numpy as np
# Define the snapshot date
SNAPSHOT = pd.Timestamp('2024-06-30')

# 1. Ensure first_order_date is in datetime format
df['first_order_date'] = pd.to_datetime(
    df['first_order_date'], errors='coerce'
)

# 2. Calculate days from first order to snapshot
df['days_since_first_order'] = (
    SNAPSHOT - df['first_order_date']
).dt.days

# 3. Fill missing values for customers who never ordered
df['days_since_first_order'] = (
    df['days_since_first_order'].fillna(0).astype(int)
)

# 4. Display the results
print(df[['first_order_date', 'days_since_first_order']].head())


  first_order_date  days_since_first_order
0              NaT                       0
1              NaT                       0
2       2023-05-22                     405
3       2022-07-02                     729
4       2022-09-18                     651


In [37]:
#6 Encoding categorical features
#  6A. ONE-HOT THE LOW-CARDINALITY COLUMNS
df = pd.get_dummies(df, columns=['plan', 'device', 'payment_method'],
                    drop_first=False, dtype=int)
print('columns after one-hot:', df.shape[1])
print([c for c in df.columns if c.startswith(('plan_', 'device_'))])
#  6B. FREQUENCY-ENCODE THE HIGH-CARDINALITY 'city'
# one-hot on 17 cities would add 17 sparse columns; frequency encoding
# keeps it to ONE numeric column = how common each city is.
freq = df['city'].value_counts(normalize=True)
df['city_freq'] = df['city'].map(freq).fillna(0)
print(df[['city', 'city_freq']].drop_duplicates().head(8))

columns after one-hot: 35
['plan_Basic', 'plan_Premium', 'plan_Standard', 'device_Desktop', 'device_Mobile', 'device_Tablet']
          city  city_freq
0      Chennai   0.103265
1    Hyderabad   0.130612
2         Pune   0.082857
4       Mumbai   0.169796
5        Delhi   0.165306
10   Bengaluru   0.159184
13       Patna   0.013469
14  Coimbatore   0.016735


In [38]:
#LAB EXCERCISE-6
import pandas as pd
# 1. Fill missing gender values with 'Unknown'
df['gender'] = df['gender'].fillna('Unknown')

# 2. One-hot encode gender
df = pd.get_dummies(
    df,
    columns=['gender'],
    prefix='gender',
    dtype=int
)

# 3. Print the new gender columns
gender_columns = [
    col for col in df.columns if col.startswith('gender_')
]
print("New gender columns:")
print(gender_columns)
print("\nEncoded gender data:")
print(df[gender_columns].head())

New gender columns:
['gender_F', 'gender_M', 'gender_Other', 'gender_Unknown']

Encoded gender data:
   gender_F  gender_M  gender_Other  gender_Unknown
0         1         0             0               0
1         1         0             0               0
2         0         0             1               0
3         1         0             0               0
4         1         0             0               0


In [39]:
#7.Rare categories & high cardinality
#  7A. BUCKET RARE CITIES INTO 'Other'
top_cities = df['city'].value_counts().head(7).index   # keep the 7 biggest
df['city_grouped'] = df['city'].where(df['city'].isin(top_cities), 'Other')
print('before:', df['city'].nunique(), 'cities')
print('after :', df['city_grouped'].nunique(), 'categories')
print(df['city_grouped'].value_counts())

before: 17 cities
after : 8 categories
city_grouped
Mumbai       416
Delhi        405
Bengaluru    390
Other        373
Hyderabad    320
Chennai      253
Pune         203
Kolkata      140
Name: count, dtype: int64


In [40]:
#LAB EXERCISE-7
import pandas as pd
# 1. Group rare cities into 'Other' if not already done
top_cities = df['city'].value_counts().head(7).index
df['city_grouped'] = df['city'].where(
    df['city'].isin(top_cities),
    'Other'
)
# 2. One-hot encode city_grouped
df = pd.get_dummies(
    df,
    columns=['city_grouped'],
    prefix='city',
    dtype=int
)
# 3. Count the new city columns
city_columns = [
    col for col in df.columns
    if col.startswith('city_')
    and col != 'city_freq'
]
print("Number of new city columns:", len(city_columns))
print("New city columns:", city_columns)
# Display the encoded data
print(df[city_columns].head())

# 4. Why group first?
# Grouping rare cities into 'Other' reduces the number of columns,
# lowers feature dimensionality, and reduces sparse features
# created by cities with very few customers.

Number of new city columns: 8
New city columns: ['city_Bengaluru', 'city_Chennai', 'city_Delhi', 'city_Hyderabad', 'city_Kolkata', 'city_Mumbai', 'city_Other', 'city_Pune']
   city_Bengaluru  city_Chennai  city_Delhi  city_Hyderabad  city_Kolkata  \
0               0             1           0               0             0   
1               0             0           0               1             0   
2               0             0           0               0             0   
3               0             0           0               0             0   
4               0             0           0               0             0   

   city_Mumbai  city_Other  city_Pune  
0            0           0          0  
1            0           0          0  
2            0           0          1  
3            0           0          1  
4            1           0          0  


In [42]:
#8.Domain-driven features
#8A. ENCODE BUSINESS KNOWLEDGE AS FEATURES
# a 'dormant' flag: no orders, or very stale
import pandas as pd
import numpy as np
# 1. Define the snapshot date
SNAPSHOT = pd.Timestamp('2024-06-30')
# 2. Ensure date columns are datetime
df['signup_date'] = pd.to_datetime(
    df['signup_date'], errors='coerce'
)
df['last_order_date'] = pd.to_datetime(
    df['last_order_date'], errors='coerce'
)
# 3. Calculate recency from the last order date
df['recency_days'] = (
    SNAPSHOT - df['last_order_date']
).dt.days
# 4. For customers without orders, use customer tenure
tenure_days = (
    SNAPSHOT - df['signup_date']
).dt.days
df['recency_days'] = df['recency_days'].fillna(tenure_days)
# 5. Create recency buckets
df['recency_bucket'] = pd.cut(
    df['recency_days'],
    bins=[-1, 30, 90, float('inf')],
    labels=['active', 'lapsing', 'dormant'],
    include_lowest=True
)
# 6. Display results
print(df[['customer_id', 'recency_days', 'recency_bucket']].head())
print("\nCustomer counts:")
print(df['recency_bucket'].value_counts(sort=False, dropna=False))


  customer_id  recency_days recency_bucket
0   CUST00001        1168.0        dormant
1   CUST00002         335.0        dormant
2   CUST00003         405.0        dormant
3   CUST00004         729.0        dormant
4   CUST00005          15.0         active

Customer counts:
recency_bucket
active      462
lapsing     543
dormant    1495
Name: count, dtype: int64


In [43]:
# LAB EXERCISE 8 — HIGH-VALUE CUSTOMER FEATURE
import numpy as np
# 1. Identify high-value customers (top 20% by monetary)
threshold = df['monetary'].quantile(0.80)
df['high_value'] = (
    df['monetary'] >= threshold
).astype(int)

# 2. Compare churn rates by high_value group
print("Monetary threshold for high-value customers:", round(threshold, 2))
print("\nCustomer counts by high_value:")
print(df['high_value'].value_counts().sort_index())
print("\nChurn rate by high_value (0 = rest, 1 = high-value):")
print(
    df.groupby('high_value')['is_churned']
      .mean()
      .round(3)
)

# 3. Another domain feature I'd add:
# purchase_trend: compare spending in the latest 3 months
# with the previous 3 months to identify declining engagement.

Monetary threshold for high-value customers: 189.87

Customer counts by high_value:
high_value
0    2000
1     500
Name: count, dtype: int64

Churn rate by high_value (0 = rest, 1 = high-value):
high_value
0    0.194
1    0.052
Name: is_churned, dtype: float64


In [44]:
#9.Assemble the feature matrix — and check for leakage
#  9A. BUILD X (features) AND y (target)
drop_cols = ['customer_id', 'signup_date', 'first_order_date', 'last_order_date',
             'city', 'gender', 'age_group', 'city_grouped', 'is_churned']
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
X = X.select_dtypes(include='number')   # keep numeric features only
y = df['is_churned']
print('Feature matrix:', X.shape)
print('All numeric? ', X.select_dtypes(exclude='number').empty)
print('Any missing? ', int(X.isna().sum().sum()))
print('\nEngineered features include:')
print([c for c in X.columns if c in ('monetary_log','recency_days','rfm_score',
       'tickets_per_order','tenure_days','city_freq','is_dormant')])

Feature matrix: (2500, 43)
All numeric?  True
Any missing?  175

Engineered features include:
['tickets_per_order', 'tenure_days', 'city_freq', 'recency_days']


In [45]:
# LAB EXERCISE 9 — SANITY CHECK & LEAKAGE DETECTION
import numpy as np
import pandas as pd
# 1. Fill missing values in X with each column's median
X = X.copy()
X = X.fillna(X.median())
# Handle any infinite values, if present
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median()).fillna(0)

# 2. Calculate feature correlations with the churn target
correlations = (
    X.assign(churn=y)
     .corr(numeric_only=True)['churn']
     .drop('churn')
     .sort_values(ascending=False)
)
print("Feature correlations with churn:")
print(correlations)

# 3. Leakage example:
# A feature such as cancellation_date or a post-churn account status
# would cause data leakage if it were unavailable at prediction time.

Feature correlations with churn:
recency_days                 0.369965
support_tickets              0.068880
tenure_days                  0.038022
city_Hyderabad               0.031792
signup_dow                   0.024051
city_Pune                    0.020868
payment_method_Card          0.020432
gender_Other                 0.018652
plan_Standard                0.015577
city_freq                    0.015418
city_Bengaluru               0.012621
email_opt_in                 0.011145
device_Mobile                0.010228
month_sin                    0.007953
gender_F                     0.007780
month_cos                    0.006207
payment_method_UPI           0.005824
signup_is_weekend            0.004310
city_Mumbai                  0.002725
gender_Unknown               0.002194
device_Tablet               -0.001560
plan_Basic                  -0.008266
city_Other                  -0.008805
plan_Premium                -0.009152
city_Delhi                  -0.009424
device_Desktop   